In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from transformers import AutoTokenizer

In [3]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arsha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
df = pd.read_csv(
    r"/Data/healthcare_dataset.csv"
)

print(df.shape)
df.head()

(112165, 3)


,instruction,input,output
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [5]:
df.columns

Index(['instruction', 'input', 'output'], dtype='object')

In [6]:
df = df.rename(columns={
    "input": "patient_query",
    "output": "doctor_response"
})

df.head()

,instruction,patient_query,doctor_response
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [7]:
df = df.dropna()

print("Dataset shape after null removal:", df.shape)

Dataset shape after null removal: (112156, 3)


In [8]:
df = df.drop_duplicates()

print("Dataset shape after duplicate removal:", df.shape)

Dataset shape after duplicate removal: (112156, 3)


In [9]:
def clean_text(text):
    
    text = str(text)
    
    # remove line breaks
    text = text.replace("\n", " ")
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text)
    
    # remove strange symbols
    text = re.sub(r"[^\w\s.,?!]", "", text)
    
    return text.strip()

In [10]:
df["patient_query"] = df["patient_query"].apply(clean_text)

df["doctor_response"] = df["doctor_response"].apply(clean_text)

df.head()

,instruction,patient_query,doctor_response
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 56 times a day for a w...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain male H...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [11]:
df = df[
    df["patient_query"].str.len() > 20
]

print(df.shape)

(111722, 3)


In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [13]:
def count_tokens(text):
    return len(tokenizer.encode(text))

In [14]:
df["query_tokens"] = df["patient_query"].apply(count_tokens)
df["response_tokens"] = df["doctor_response"].apply(count_tokens)

df.head()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


,instruction,patient_query,doctor_response,query_tokens,response_tokens
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most...",143,135
1,"If you are a doctor, please answer the medical...",My baby has been pooing 56 times a day for a w...,Hi... Thank you for consulting in Chat Doctor....,74,115
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ...",69,179
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain male H...,HI. You have two different problems. The lump ...,65,47
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...,65,122


In [15]:
print(df["query_tokens"].describe())
print(df["response_tokens"].describe())

count    111722.000000
mean        102.130869
std          56.899711
min           4.000000
25%          71.000000
50%          86.000000
75%         116.000000
max        2348.000000
Name: query_tokens, dtype: float64
count    111722.000000
mean        131.467365
std          51.900642
min           3.000000
25%         101.000000
50%         123.000000
75%         155.000000
max         678.000000
Name: response_tokens, dtype: float64


In [16]:
medical_categories = {
    "cardiology": [
        "heart",
        "chest pain",
        "blood pressure",
        "cardiac"
    ],
    
    "neurology": [
        "brain",
        "headache",
        "migraine",
        "seizure"
    ],
    
    "dermatology": [
        "skin",
        "rash",
        "itching",
        "acne"
    ],
    
    "orthopedic": [
        "bone",
        "joint",
        "knee",
        "fracture"
    ],
    
    "general_medicine": []
}

In [17]:
def assign_category(text):
    
    text = text.lower()
    
    for category, keywords in medical_categories.items():
        
        for keyword in keywords:
            
            if keyword in text:
                return category
    
    return "general_medicine"

In [18]:
df["medical_category"] = df["patient_query"].apply(assign_category)

df.head()

,instruction,patient_query,doctor_response,query_tokens,response_tokens,medical_category
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most...",143,135,general_medicine
1,"If you are a doctor, please answer the medical...",My baby has been pooing 56 times a day for a w...,Hi... Thank you for consulting in Chat Doctor....,74,115,dermatology
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ...",69,179,general_medicine
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain male H...,HI. You have two different problems. The lump ...,65,47,general_medicine
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...,65,122,general_medicine


In [19]:
df["medical_category"].value_counts()

medical_category
general_medicine    79708
cardiology          11114
dermatology          8933
orthopedic           6494
neurology            5473
Name: count, dtype: int64

In [20]:
def retrieval_chunk(row):
    
    return f"""
    Patient Query:
    {row['patient_query']}
    
    Doctor Response:
    {row['doctor_response']}
    """

In [21]:
df["retrieval_text"] = df.apply(
    retrieval_chunk,
    axis=1
)

df[[
    "retrieval_text",
    "medical_category"
]].head()

,retrieval_text,medical_category
0,\n Patient Query:\n I woke up this morni...,general_medicine
1,\n Patient Query:\n My baby has been poo...,dermatology
2,"\n Patient Query:\n Hello, My husband is...",general_medicine
3,\n Patient Query:\n lump under left nipp...,general_medicine
4,\n Patient Query:\n I have a 5 month old...,general_medicine


In [22]:
processed_df = df[[
    "retrieval_text",
    "patient_query",
    "doctor_response",
    "medical_category",
    "query_tokens",
    "response_tokens"
]]

processed_df.head()

,retrieval_text,patient_query,doctor_response,medical_category,query_tokens,response_tokens
0,\n Patient Query:\n I woke up this morni...,I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most...",general_medicine,143,135
1,\n Patient Query:\n My baby has been poo...,My baby has been pooing 56 times a day for a w...,Hi... Thank you for consulting in Chat Doctor....,dermatology,74,115
2,"\n Patient Query:\n Hello, My husband is...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ...",general_medicine,69,179
3,\n Patient Query:\n lump under left nipp...,lump under left nipple and stomach pain male H...,HI. You have two different problems. The lump ...,general_medicine,65,47
4,\n Patient Query:\n I have a 5 month old...,I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...,general_medicine,65,122


In [23]:
processed_df = df[[
    "retrieval_text",
    "patient_query",
    "doctor_response",
    "query_tokens",
    "response_tokens"
]]

processed_df.head()

,retrieval_text,patient_query,doctor_response,query_tokens,response_tokens
0,\n Patient Query:\n I woke up this morni...,I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most...",143,135
1,\n Patient Query:\n My baby has been poo...,My baby has been pooing 56 times a day for a w...,Hi... Thank you for consulting in Chat Doctor....,74,115
2,"\n Patient Query:\n Hello, My husband is...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ...",69,179
3,\n Patient Query:\n lump under left nipp...,lump under left nipple and stomach pain male H...,HI. You have two different problems. The lump ...,65,47
4,\n Patient Query:\n I have a 5 month old...,I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...,65,122


In [25]:
processed_df.to_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\processed_healthcare_dataset.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
